# C1 full fine-tuning

**Later illustrated companion; not the executed training notebook.** Historical metrics below are quoted from copied run records, never represented as notebook outputs.

The local dev score uses a threshold recorded in the run; inspect its calibration/selection record before treating it as independent model-selection evidence. A prepared model notebook is source scaffolding, while `v3_t4x2.ipynb` and the Kaggle run evidence identify the executed v3 lineage.

## Task and architecture

The C1 recurrent hybrid combines a ten-layer encoder with a four-layer recurrent decoder and cross-attention. Full tuning updates its backbone and the shared evidence head.

**Adaptation:** All backbone weights trainable

Data path: official train → training-only fit/selection → official labeled dev report → 1,517-label-free test predictions. The original `code/` and `evidence/` directories contain the audit trail.

## Recorded result

Original Kaggle run: complete; 899 optimizer updates; selected step 899. Official 994-case dev macro-F1 **0.484079** at recorded threshold 0.437448; default 0.5 dev macro-F1 0.351393. Verified prediction ZIP SHA-256 `4d32491e98558bf8e0bd240519c140191a7faac18c3a23fa8feb80659b4e022a`, exactly 1,517 IDs.

Local official dev, calibration CV, and Codabench scores are different measurements. A historical score is not an output of this notebook.

## Source excerpt: `models.py`

The complete source is in `code/`. This excerpt illustrates the exact preserved implementation; it is not a training replay.

In [ ]:
"""Four original backbones and six task adaptation strategies.

LoRMA uses an input-side multiplicative low-rank update. Adapters add a
bottleneck residual to FFN output projections. No external pretrained weights.
"""
from pathlib import Path
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from phase3_rr.models import load_backbone,RRClassifier,adapt

class OutputAdapter(nn.Module):
 def __init__(self,base,bottleneck=32):
  super().__init__();self.base=base
  self.down=nn.Linear(base.out_features,bottleneck)
  self.up=nn.Linear(bottleneck,base.out_features)
  nn.init.zeros_(self.up.weight);nn.init.zeros_(self.up.bias)
 def forward(self,x):
  y=self.base(x);return y+self.up(F.gelu(self.down(y)))

def configure(backbone,family,method,rank=8,alpha=16):
 if method=='full':backbone.requires_grad_(True)
 elif method=='lorma':adapt(backbone,'lorma',rank,alpha)
 elif method=='bitfit':
  backbone.requires_grad_(False)
  for name,p in backbone.named_parameters():
   if name.rsplit('.',1)[-1].startswith('bias'):p.requires_grad_(True)
 elif method=='partial':
  if family!='b1':raise ValueError('This partial experiment is defined for B1')
  backbone.requires_grad_(False)
  for block in list(backbone.decoder)[-2:]+list(backbone.encoder)[-2:]:block.requires_grad_(True)
  backbone.decoder_norm.requires_grad_(True)
  if hasattr(backbone,'encoder_norm'):backbone.encoder_norm.requires_grad_(True)
 elif method=='adapters':
  backbone.requires_grad_(False)
  def walk(module):
   for name,child in list(module.named_children()):
    if name=='w2' and isinstance(child,nn.Linear):setattr(module,name,OutputAdapter(child))
    else:walk(child)
  walk(backbone)
 else:raise ValueError('Unknown adaptation method '+method)
 names=[n for n,p in backbone.named_parameters() if p.requires_grad]
 if not names:raise ValueError('No original-backbone/adaptation parameters trainable')
 return names

class CaseModel(nn.Module):
 def __init__(self,backbone,family,width):
  super().__init__();self.backbone=backbone;self.family=family;self.width=width
  self.pool=nn.Sequential(nn.Linear(width+2,64),nn.Tanh(),nn.Linear(64,1))
  self.head=nn.Sequential(nn.LayerNorm(width),nn.Dropout(.15),nn.Linear(width,2))
 def token_features(self,ids):return RRClassifier.representations(self,ids)
 def chunk_feature(self,ids):
  features=self.token_features(ids)
  # Chunks are unpadded, each encoded separately; all original tokens participate.
  return features.float().mean(1)
 def forward(self,chunks,positions,device):
  values=[]
  for ids in chunks:
   x=torch.as_tensor([ids],dtype=torch.long,device=device)
   if self.training:
    # Non-reentrant recomputation also preserves gradients with frozen embeddings.
    value=checkpoint(self.chunk_feature,x,use_reentrant=False)
   else:value=self.chunk_feature(x)
   values.append(value)
  h=torch.cat(values,0)
  pos=torch.as_tensor(positions,dtype=h.dtype,device=device).reshape(-1,1)
  scores=self.pool(torch.cat((h,pos,1-pos),-1)).float().squeeze(-1)
  weights=scores.softmax(0)
  return self.head((h*weights[:,None]).sum(0,keepdim=True))

class EvidenceCaseModel(CaseModel):
 """Learned token evidence and ordered chunk context; own backbone stays intact."""
 def __init__(self,backbone,family,width):
  super().__init__(backbone,family,width)
  hidden=min(128,max(4,width//2))
  self.token_norm=nn.LayerNorm(width)
  self.token_score=nn.Sequential(nn.Linear(width,min(64,width)),nn.Tanh(),nn.Linear(min(64,width),1))
  nn.init.zeros_(self.token_score[-1].weight);nn.init.zeros_(self.token_score[-1].bias)
  self.token_mix=nn.Parameter(torch.zeros(3))
  self.position=nn.Linear(2,width,bias=False);nn.init.zeros_(self.position.weight)
  self.chunk_context=nn.GRU(width,hidden,bidirectional=True,batch_first=True)
  self.context_projection=nn.Linear(2*hidden,width)
  self.context_scale=nn.Parameter(torch.tensor(.1))
  self.context_norm=nn.LayerNorm(width)

## Reproduction boundary

The package has code, configs, selected records, and prediction ZIP. Original model weights and official data are retained in the local archives, not duplicated here. Paths in the original configs must be remapped before an authorized rerun.